# 🎨 ComfyUI Colab — WAI-illustrious + YOLO + SAM

**Thứ tự chạy:** Cell **1 → 1B → 2 → 3**  (đã tách vì Cell 1 gộp YOLO hay bị “treo” im lặng)

1. Runtime → Change runtime type → **T4 GPU** → Save
2. Cell 1 xong khi thấy `✅ Xong Cell 1` (~2–3 phút, **sẽ có chữ in từng bước**)
3. Cell 1B cài Impact Pack (~30–60 giây)
4. Cell 2 dán Civitai key, tải WAI + YOLO + SAM
5. Cell 3 lấy link → dán tab mới → kéo `WAI_ChiTietNho.json` → **R** → chọn model

Nếu Cell 1 đứng ở `Mounting Drive` quá 1 phút: cửa sổ xin quyền bị chặn → cho phép popup, hoặc tick **Bỏ qua Drive** rồi chạy lại.

In [ ]:
# ===== CELL 1: Drive + ComfyUI (KHÔNG cài YOLO ở đây — tránh treo pip) =====
BO_QUA_DRIVE = False  # @param {type:"boolean"}

import os, time, sys
def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

log('GPU:')
!nvidia-smi --query-gpu=name,memory.total --format=csv

USE_DRIVE = False
if not BO_QUA_DRIVE:
    log('Kết nối Drive — nếu đứng >60s: tick BO_QUA_DRIVE, Runtime → Interrupt, chạy lại.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')  # KHÔNG force_remount (dễ treo)
        USE_DRIVE = True
        log('Drive OK')
    except Exception as e:
        log(f'Drive lỗi ({e}) → ổ tạm')
else:
    log('Bỏ qua Drive (ổ tạm, model mất khi ngắt phiên)')

ROOT = '/content/drive/MyDrive/AI_Models' if USE_DRIVE else '/content/AI_Models'
for p in [f'{ROOT}/checkpoints', f'{ROOT}/ultralytics/bbox', f'{ROOT}/sams']:
    os.makedirs(p, exist_ok=True)
CKPT_DIR, YOLO_DIR, SAM_DIR = f'{ROOT}/checkpoints', f'{ROOT}/ultralytics/bbox', f'{ROOT}/sams'
open('/content/ckpt_dir.txt','w').write(CKPT_DIR)
open('/content/yolo_dir.txt','w').write(YOLO_DIR)
open('/content/sam_dir.txt','w').write(SAM_DIR)
open('/content/root_dir.txt','w').write(ROOT)

os.chdir('/content')
if os.path.isdir('/content/ComfyUI') and os.path.isfile('/content/ComfyUI/main.py'):
    log('ComfyUI đã có — không clone lại')
else:
    log('Clone ComfyUI (~1–2 phút, có % là chưa chết)...')
    !git clone --progress --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

os.chdir('/content/ComfyUI')
log('Cài thư viện ComfyUI (giữ PyTorch GPU của Colab, ~1 phút)...')
!grep -viE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' requirements.txt > /content/req_notorch.txt
!pip install -r /content/req_notorch.txt

import torch
assert torch.cuda.is_available(), '❌ Không thấy GPU. Runtime → Change runtime type → T4 GPU → Restart session → Cell 1'
log(f'PyTorch {torch.__version__} | GPU {torch.cuda.get_device_name(0)}')

log('✅ Xong Cell 1 — chạy tiếp Cell 1B')

In [ ]:
# ===== CELL 1B: Impact Pack + YOLO/SAM (nhẹ, không đè OpenCV/Torch) =====
import os, time, sys, subprocess
def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

ROOT = open('/content/root_dir.txt').read().strip()
CKPT_DIR = open('/content/ckpt_dir.txt').read().strip()
YOLO_DIR = open('/content/yolo_dir.txt').read().strip()
SAM_DIR  = open('/content/sam_dir.txt').read().strip()
CN = '/content/ComfyUI/custom_nodes'
os.makedirs(CN, exist_ok=True)
os.chdir(CN)

def clone(url, folder):
    path = os.path.join(CN, folder)
    if os.path.isdir(path) and os.listdir(path):
        log(f'{folder} đã có — bỏ qua clone')
        return
    log(f'Clone {folder}...')
    r = subprocess.run(['git','clone','--progress','--depth','1', url, path])
    if r.returncode != 0:
        raise RuntimeError(f'Clone {folder} thất bại')

clone('https://github.com/ltdrdata/ComfyUI-Impact-Pack.git', 'ComfyUI-Impact-Pack')
clone('https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git', 'ComfyUI-Impact-Subpack')

log('pip nhẹ: piexif dill segment-anything + ultralytics --no-deps (không cài lại opencv/torch)')
!pip install piexif dill segment-anything
!pip install ultralytics --no-deps

log('Symlink checkpoints / YOLO / SAM')
pairs = [
    ('/content/ComfyUI/models/checkpoints', CKPT_DIR),
    ('/content/ComfyUI/models/ultralytics', f'{ROOT}/ultralytics'),
    ('/content/ComfyUI/models/sams', SAM_DIR),
]
for path, dest in pairs:
    os.makedirs(dest, exist_ok=True)
    if os.path.islink(path) or os.path.exists(path):
        if os.path.islink(path) or os.path.isdir(path):
            !rm -rf "{path}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    os.symlink(dest, path)
    log(f'  {path} → {dest}')

print()
!ls /content/ComfyUI/custom_nodes | grep -i impact
log('✅ Xong Cell 1B — chạy Cell 2 (tải file .pt / .safetensors)')

In [ ]:
# ===== CELL 2: Tải WAI + YOLO mặt/tay + SAM =====
CIVITAI_KEY = ""  # @param {type:"string"}

import os, time, sys
def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True); sys.stdout.flush()

CKPT_DIR = open('/content/ckpt_dir.txt').read().strip()
YOLO_DIR = open('/content/yolo_dir.txt').read().strip()
SAM_DIR  = open('/content/sam_dir.txt').read().strip()
os.makedirs(YOLO_DIR, exist_ok=True)
os.makedirs(SAM_DIR, exist_ok=True)
log(f'Checkpoint {CKPT_DIR}')
log(f'YOLO {YOLO_DIR}')
log(f'SAM  {SAM_DIR}')

def download(url, path, min_bytes, label):
    if os.path.exists(path) and os.path.getsize(path) > min_bytes:
        log(f'✅ {label} đã có ({os.path.getsize(path)/1e6:.0f} MB)')
        return
    log(f'⬇️  {label}')
    !wget --show-progress -c -O "{path}" "{url}"
    size = os.path.getsize(path) if os.path.exists(path) else 0
    if size < min_bytes:
        if os.path.exists(path): os.remove(path)
        raise RuntimeError(f'❌ {label} thất bại ({size} byte)')
    log(f'✅ {label} ({size/1e6:.0f} MB)')

WAI = f'{CKPT_DIR}/WAI-illustrious.safetensors'
if os.path.exists(WAI) and os.path.getsize(WAI) > 6_000_000_000:
    log('✅ WAI-illustrious đã có')
else:
    assert CIVITAI_KEY.strip(), '❌ Dán Civitai API key vào ô CIVITAI_KEY'
    download(
        f'https://civitai.com/api/download/models/2514310?token={CIVITAI_KEY.strip()}',
        WAI, 6_000_000_000, 'WAI-illustrious')

download('https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt',
         f'{YOLO_DIR}/face_yolov8m.pt', 20_000_000, 'YOLO mặt')
download('https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8s.pt',
         f'{YOLO_DIR}/hand_yolov8s.pt', 8_000_000, 'YOLO tay')
download('https://huggingface.co/Claquasse/foot_anime_yolo/resolve/main/foot_anime_yolo11m_v3.pt',
         f'{YOLO_DIR}/foot_anime_yolo11m_v3.pt', 15_000_000, 'YOLO chân anime v3')

SAM = f'{SAM_DIR}/sam_vit_b_01ec64.pth'
try:
    download('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
             SAM, 300_000_000, 'SAM ViT-B')
except Exception as e:
    log(f'Facebook lỗi ({e}), thử HF')
    download('https://huggingface.co/ybelkada/segment-anything/resolve/main/checkpoints/sam_vit_b_01ec64.pth',
             SAM, 300_000_000, 'SAM ViT-B (HF)')

!ls -lh {CKPT_DIR}
!ls -lh {YOLO_DIR}
!ls -lh {SAM_DIR}
log('✅ Cell 2 xong — chạy Cell 3')

In [ ]:
# ===== CELL 3: ComfyUI nền + link cloudflared =====
import subprocess, time, socket, re, os, sys
def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True); sys.stdout.flush()

!pkill -f "python main.py" 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

if not os.path.exists('/usr/local/bin/cloudflared'):
    log('Cài cloudflared...')
    !wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

os.chdir('/content/ComfyUI')
comfy_log = open('/content/comfyui.log', 'w')
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--enable-cors-header'],
    stdout=comfy_log, stderr=subprocess.STDOUT)
log('Khởi động ComfyUI (30–90s)...')

for i in range(180):
    time.sleep(1)
    if i in (15, 30, 60, 90):
        log(f'  ... {i}s')
    if comfy.poll() is not None:
        raise RuntimeError('❌ ComfyUI tắt. Chạy: !tail -40 /content/comfyui.log')
    try:
        with socket.create_connection(('127.0.0.1', 8188), timeout=1):
            break
    except OSError:
        pass
else:
    raise RuntimeError('❌ Quá 3 phút. !tail -40 /content/comfyui.log')
log('ComfyUI đã chạy')

cf_log = open('/content/cloudflared.log', 'w')
subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188',
     '--http-host-header', '127.0.0.1:8188', '--protocol', 'http2'],
    stdout=cf_log, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    t = open('/content/cloudflared.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', t)
    if m:
        url = m.group(0)
        break
print('\n' + '=' * 60)
if url:
    print('🎨 COPY LINK, DÁN VÀO THANH ĐỊA CHỈ TAB MỚI:')
    print(url)
else:
    print('⚠️ Chưa có link — dùng Cell 5')
print('=' * 60)
print('Kéo WAI_ChiTietNho.json → R → chọn WAI-illustrious')

In [ ]:
# ===== CELL 3B (TÙY CHỌN): Giao diện gọn tiếng Việt =====
!pip install -q gradio websocket-client
!wget -q -O /content/giaodien_tao_anh.py https://raw.githubusercontent.com/caone1196-sketch/t-i-li-u/arena/01a09a8b-t-i-li-u/giaodien_tao_anh.py
print('Đợi link https://xxxx.gradio.live — đừng dừng cell khi đang dùng.\n')
!python /content/giaodien_tao_anh.py

In [ ]:
# ===== CELL 4: Kiểm tra =====
!curl -s -o /dev/null -w "A) ComfyUI: HTTP %{http_code}\n" --max-time 20 http://127.0.0.1:8188/system_stats
import re, os
txt = open('/content/cloudflared.log').read() if os.path.exists('/content/cloudflared.log') else ''
m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
print('Link:', m.group(0) if m else '(chưa có)')
print('\nImpact:'); !ls /content/ComfyUI/custom_nodes | grep -i impact || echo THIEU
print('\nYOLO:'); !ls -lh /content/ComfyUI/models/ultralytics/bbox/ 2>/dev/null || echo THIEU
print('\nSAM:'); !ls -lh /content/ComfyUI/models/sams/ 2>/dev/null || echo THIEU
print('\nlog:'); !tail -30 /content/comfyui.log

In [ ]:
# ===== CELL 5: localtunnel dự phòng =====
!npm install -g localtunnel > /dev/null 2>&1
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print('🔑 Tunnel Password:', ip)
!lt --port 8188

## 🔧 Cell 1 bị treo — đã sửa gì

Bản trước gộp `pip install ultralytics opencv-python-headless` (im lặng `-q`) → Colab đứng 10–20 phút như chết, đôi khi đè OpenCV/Torch.

Bây giờ:
- Cell 1 chỉ Drive + ComfyUI, **in timestamp từng bước**
- **Không** `force_remount` Drive
- **Không** xóa ComfyUI nếu đã clone
- Cell 1B: Impact + `ultralytics --no-deps` (không cài lại opencv/torch)

**Đứng ở Mounting Drive:** tick `BO_QUA_DRIVE` → Interrupt → chạy lại Cell 1.

**Đứng ở Clone ComfyUI không có %:** mạng GitHub chậm. Interrupt, chạy lại (sẽ resume nếu thư mục lỗi thì xóa `/content/ComfyUI` tay).

Workflow: `WAI_ChiTietNho.json`. Node vẫn đỏ → chưa chạy Cell 1B.